<a href="https://colab.research.google.com/github/amzad-786githumb/Privacy-Preserving-Synthetic-Tabular-Data-Generation-Using-Generative-Adversarial-Networks/blob/main/05_Proposed_SPP_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PART 5.1.1 — INSTALL REQUIRED PACKAGES
# ==============================================================================

print("=" * 100)
print("PART 5.1.1 — INSTALL REQUIRED PACKAGES")
print("=" * 100)

import sys
import subprocess

def install(package):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package]
    )

packages = [

    "sdv>=1.18.0",
    "ctgan>=0.10.0",
    "torch",
    "torchvision",
    "torchaudio",
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "scipy",
    "tqdm",
    "joblib",
    "pyyaml",
    "openpyxl"

]

for pkg in packages:

    try:

        __import__(pkg.split(">=")[0].replace("-", "_"))

    except:

        install(pkg)

print("✓ All required packages installed")

PART 5.1.1 — INSTALL REQUIRED PACKAGES
✓ All required packages installed


In [2]:
# ==============================================================================
# PART 5.1.2 — IMPORT LIBRARIES
# ==============================================================================

print("=" * 100)
print("PART 5.1.2 — IMPORT LIBRARIES")
print("=" * 100)

import os
import gc
import time
import random
import warnings
import platform

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import yaml
import joblib

from tqdm.auto import tqdm

import torch

from sklearn.model_selection import train_test_split

from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

import sdv

print("✓ Libraries Imported Successfully")# ============================================================
# Cell A1 : Import Libraries
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset

from sdv.metadata import SingleTableMetadata

print("Libraries imported successfully.")

PART 5.1.2 — IMPORT LIBRARIES
✓ Libraries Imported Successfully
Libraries imported successfully.


In [3]:
# ==============================================================================
# PART 5.1.3 — GPU CONFIGURATION & REPRODUCIBILITY
# ==============================================================================

print("=" * 100)
print("PART 5.1.3 — GPU CONFIGURATION")
print("=" * 100)

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

RANDOM_STATE = 42

random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():

    torch.cuda.manual_seed(RANDOM_STATE)

    torch.cuda.manual_seed_all(RANDOM_STATE)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device : {DEVICE}")

if torch.cuda.is_available():

    print(torch.cuda.get_device_name(0))

print(f"Random Seed : {RANDOM_STATE}")

PART 5.1.3 — GPU CONFIGURATION
Device : cpu
Random Seed : 42


In [4]:
# ==============================================================================
# PART 5.1.4 — NOTEBOOK CONFIGURATION
# ==============================================================================

print("=" * 100)
print("PART 5.1.4 — NOTEBOOK CONFIGURATION")
print("=" * 100)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

pd.set_option("display.max_rows", 200)

pd.set_option("display.width", 200)

pd.set_option(

    "display.float_format",

    lambda x: f"{x:.4f}"

)

sns.set_theme(

    style="whitegrid",

    context="notebook"

)

plt.rcParams["figure.figsize"] = (10,6)

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print("✓ Notebook configuration completed")

PART 5.1.4 — NOTEBOOK CONFIGURATION
✓ Notebook configuration completed


In [5]:
# ==============================================================================
# PART 5.1.5 — ENVIRONMENT SUMMARY
# ==============================================================================

print("=" * 100)
print("PART 5.1.5 — ENVIRONMENT SUMMARY")
print("=" * 100)

ENVIRONMENT_DF = pd.DataFrame({

    "Component":[

        "Python",

        "PyTorch",

        "SDV",

        "Device",

        "Random Seed"

    ],

    "Value":[

        platform.python_version(),

        torch.__version__,

        sdv.__version__,

        str(DEVICE),

        RANDOM_STATE

    ]

})

display(ENVIRONMENT_DF)

print("\n✓ PART 5.1 COMPLETED SUCCESSFULLY")
print("Ready for PART 5.2 — Load Project Configuration")

PART 5.1.5 — ENVIRONMENT SUMMARY


,Component,Value
0,Python,3.12.13
1,PyTorch,2.11.0+cpu
2,SDV,1.37.3
3,Device,cpu
4,Random Seed,42



✓ PART 5.1 COMPLETED SUCCESSFULLY
Ready for PART 5.2 — Load Project Configuration


In [23]:
# ==============================================================================
# PART 5.2.1 — LOAD PROJECT CONFIGURATION
# ==============================================================================

import os
import yaml
import pandas as pd

CONFIG_FILE = "/content/drive/MyDrive/SPP_GAN_Project/config.yaml"

with open(CONFIG_FILE, "r") as file:

    CONFIG = yaml.safe_load(file)

print("="*100)
print("PART 5.2.1 — PROJECT CONFIGURATION")
print("="*100)

display(

    pd.DataFrame(

        CONFIG.items(),

        columns=["Parameter","Value"]

    )

)

PART 5.2.1 — PROJECT CONFIGURATION


,Parameter,Value
0,batch_size,256
1,beta1,0.5000
2,beta2,0.9990
3,delta,0.0000
4,device,cuda
5,epochs,300
6,gradient_clip,1.0000
7,latent_dimension,128
8,learning_rate,0.0002
9,noise_multiplier,1.1000


In [24]:
# ==============================================================================
# PART 5.2.2 — LOAD TRAINING PARAMETERS
# ==============================================================================

SEED = CONFIG["seed"]

DEVICE = CONFIG["device"]

EPOCHS = CONFIG["epochs"]

BATCH_SIZE = CONFIG["batch_size"]

LATENT_DIM = CONFIG["latent_dimension"]

LEARNING_RATE = CONFIG["learning_rate"]

OPTIMIZER = CONFIG["optimizer"]

BETA1 = CONFIG["beta1"]

BETA2 = CONFIG["beta2"]

print("="*100)
print("TRAINING PARAMETERS")
print("="*100)

print(f"Seed            : {SEED}")
print(f"Device          : {DEVICE}")
print(f"Epochs          : {EPOCHS}")
print(f"Batch Size      : {BATCH_SIZE}")
print(f"Latent Dim      : {LATENT_DIM}")
print(f"Learning Rate   : {LEARNING_RATE}")
print(f"Optimizer       : {OPTIMIZER}")
print(f"Beta1           : {BETA1}")
print(f"Beta2           : {BETA2}")

TRAINING PARAMETERS
Seed            : 42
Device          : cuda
Epochs          : 300
Batch Size      : 256
Latent Dim      : 128
Learning Rate   : 0.0002
Optimizer       : Adam
Beta1           : 0.5
Beta2           : 0.999


In [25]:
# ==============================================================================
# PART 5.2.3 — DEFINE PROJECT DIRECTORIES
# ==============================================================================

PROJECT_ROOT = "/content/drive/MyDrive/SPP_GAN_Project"

DATASET_DIR = os.path.join(

    PROJECT_ROOT,

    "datasets"

)

PROCESSED_DATA_DIR = os.path.join(

    PROJECT_ROOT,

    "datasets",

    "processed"

)

MODEL_DIR = os.path.join(

    PROJECT_ROOT,

    "models"

)

RESULT_DIR = os.path.join(

    PROJECT_ROOT,

    "results"

)

REPORT_DIR = os.path.join(

    PROJECT_ROOT,

    "reports"

)

TVAE_MODEL_DIR = os.path.join(

    MODEL_DIR,

    "TVAE"

)

TVAE_RESULT_DIR = os.path.join(

    RESULT_DIR,

    "TVAE"

)

TVAE_REPORT_DIR = os.path.join(

    REPORT_DIR,

    "TVAE"

)

print("="*100)
print("PROJECT DIRECTORIES")
print("="*100)

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Processed Data    : {PROCESSED_DATA_DIR}")
print(f"Model Directory   : {TVAE_MODEL_DIR}")
print(f"Result Directory  : {TVAE_RESULT_DIR}")
print(f"Report Directory  : {TVAE_REPORT_DIR}")

PROJECT DIRECTORIES
Project Root      : /content/drive/MyDrive/SPP_GAN_Project
Processed Data    : /content/drive/MyDrive/SPP_GAN_Project/datasets/processed
Model Directory   : /content/drive/MyDrive/SPP_GAN_Project/models/TVAE
Result Directory  : /content/drive/MyDrive/SPP_GAN_Project/results/TVAE
Report Directory  : /content/drive/MyDrive/SPP_GAN_Project/reports/TVAE


In [26]:
# ==============================================================================
# PART 5.2.4 — CREATE PROJECT DIRECTORIES
# ==============================================================================

os.makedirs(

    TVAE_MODEL_DIR,

    exist_ok=True

)

os.makedirs(

    TVAE_RESULT_DIR,

    exist_ok=True

)

os.makedirs(

    TVAE_REPORT_DIR,

    exist_ok=True

)

print("="*100)
print("TVAE DIRECTORIES CREATED")
print("="*100)

print(TVAE_MODEL_DIR)

print(TVAE_RESULT_DIR)

print(TVAE_REPORT_DIR)

TVAE DIRECTORIES CREATED
/content/drive/MyDrive/SPP_GAN_Project/models/TVAE
/content/drive/MyDrive/SPP_GAN_Project/results/TVAE
/content/drive/MyDrive/SPP_GAN_Project/reports/TVAE


In [27]:
# ==============================================================================
# PART 5.2.5 — VERIFY PROJECT CONFIGURATION
# ==============================================================================

EXPECTED_DATASETS = [

    "adult_income_processed.csv",

    "bank_marketing_processed.csv",

    "breast_cancer_processed.csv"

]

processed_ok = all(

    os.path.isfile(

        os.path.join(

            PROCESSED_DATA_DIR,

            file

        )

    )

    for file in EXPECTED_DATASETS

)

verification = {

    "Configuration Loaded":

        CONFIG is not None,

    "Processed Data":

        processed_ok,

    "TVAE Model Directory":

        os.path.isdir(

            TVAE_MODEL_DIR

        ),

    "TVAE Result Directory":

        os.path.isdir(

            TVAE_RESULT_DIR

        ),

    "TVAE Report Directory":

        os.path.isdir(

            TVAE_REPORT_DIR

        )

}

VERIFY_DF = pd.DataFrame({

    "Component":

        verification.keys(),

    "Status":

        verification.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x:

        "PASS"

        if x

        else "FAIL"

)

print("="*100)
print("PART 5.2 VERIFICATION")
print("="*100)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n"+"="*100)

    print("✓ PART 5.2 COMPLETED SUCCESSFULLY")

    print("Ready for Part 5.3 — Load Processed Datasets")

    print("="*100)

else:

    print("\nFAILED COMPONENTS\n")

    display(

        VERIFY_DF[

            VERIFY_DF["Status"] == False

        ]

    )

    raise RuntimeError(

        "Configuration verification failed."

    )

PART 5.2 VERIFICATION


,Component,Status,Result
0,Configuration Loaded,True,PASS
1,Processed Data,True,PASS
2,TVAE Model Directory,True,PASS
3,TVAE Result Directory,True,PASS
4,TVAE Report Directory,True,PASS



✓ PART 5.2 COMPLETED SUCCESSFULLY
Ready for Part 5.3 — Load Processed Datasets


In [28]:
# ==============================================================================
# PART 5.3.1 — INITIALIZE DATASET LOADING
# ==============================================================================

print("=" * 100)
print("PART 5.3 — LOAD PROCESSED DATASETS")
print("=" * 100)

DATASETS = [

    "adult_income",

    "bank_marketing",

    "breast_cancer"

]

DATA_FILES = {

    "adult_income":

        "adult_income_processed.csv",

    "bank_marketing":

        "bank_marketing_processed.csv",

    "breast_cancer":

        "breast_cancer_processed.csv"

}

print("Datasets to Load\n")

for dataset in DATASETS:

    print(f"• {dataset}")

PART 5.3 — LOAD PROCESSED DATASETS
Datasets to Load

• adult_income
• bank_marketing
• breast_cancer


In [29]:
# ==============================================================================
# PART 5.3.2 — LOAD PROCESSED DATASETS
# ==============================================================================

DATA_DICT = {}

LOAD_SUMMARY = []

for dataset in DATASETS:

    file_path = os.path.join(

        PROCESSED_DATA_DIR,

        DATA_FILES[dataset]

    )

    df = pd.read_csv(file_path)

    DATA_DICT[dataset] = df

    LOAD_SUMMARY.append({

        "Dataset":

            dataset,

        "Rows":

            df.shape[0],

        "Columns":

            df.shape[1],

        "Loaded":

            True

    })

LOAD_SUMMARY_DF = pd.DataFrame(

    LOAD_SUMMARY

)

print("=" * 100)
print("DATASETS LOADED")
print("=" * 100)

display(LOAD_SUMMARY_DF)

DATASETS LOADED


,Dataset,Rows,Columns,Loaded
0,adult_income,32510,15,True
1,bank_marketing,45211,17,True
2,breast_cancer,569,32,True


In [30]:
# ==============================================================================
# PART 5.3.3 — DATASET OVERVIEW
# ==============================================================================

OVERVIEW = []

for dataset, df in DATA_DICT.items():

    OVERVIEW.append({

        "Dataset":

            dataset,

        "Rows":

            len(df),

        "Columns":

            len(df.columns),

        "Missing Values":

            int(df.isna().sum().sum()),

        "Duplicate Rows":

            int(df.duplicated().sum())

    })

OVERVIEW_DF = pd.DataFrame(

    OVERVIEW

)

print("=" * 100)
print("DATASET OVERVIEW")
print("=" * 100)

display(OVERVIEW_DF)

DATASET OVERVIEW


,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,adult_income,32510,15,0,0
1,bank_marketing,45211,17,0,0
2,breast_cancer,569,32,0,0


In [31]:
# ==============================================================================
# PART 5.3.4 — STORE DATASET METADATA
# ==============================================================================

DATASET_METADATA = {}

for dataset, df in DATA_DICT.items():

    DATASET_METADATA[dataset] = {

        "Rows":

            df.shape[0],

        "Columns":

            df.shape[1],

        "Categorical Columns":

            list(

                df.select_dtypes(

                    include=["object","category"]

                ).columns

            ),

        "Numerical Columns":

            list(

                df.select_dtypes(

                    exclude=["object","category"]

                ).columns

            )

    }

print("=" * 100)
print("DATASET METADATA CREATED")
print("=" * 100)

print(f"Datasets Stored : {len(DATASET_METADATA)}")

DATASET METADATA CREATED
Datasets Stored : 3


In [32]:
# ==============================================================================
# PART 5.3.5 — VERIFICATION
# ==============================================================================

verification = {

    "Datasets Loaded":

        len(DATA_DICT) == len(DATASETS),

    "Adult Income":

        "adult_income" in DATA_DICT,

    "Bank Marketing":

        "bank_marketing" in DATA_DICT,

    "Breast Cancer":

        "breast_cancer" in DATA_DICT,

    "Metadata Created":

        len(DATASET_METADATA) == len(DATASETS)

}

VERIFY_DF = pd.DataFrame({

    "Component":

        verification.keys(),

    "Status":

        verification.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x:

        "PASS"

        if x

        else "FAIL"

)

print("=" * 100)
print("PART 5.3 VERIFICATION")
print("=" * 100)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n" + "=" * 100)

    print("✓ PART 5.3 COMPLETED SUCCESSFULLY")

    print("Ready for Part 5.4 — Automatic Metadata Detection")

    print("=" * 100)

else:

    display(

        VERIFY_DF[

            VERIFY_DF["Status"] == False

        ]

    )

    raise RuntimeError(

        "Dataset loading verification failed."

    )

PART 5.3 VERIFICATION


,Component,Status,Result
0,Datasets Loaded,True,PASS
1,Adult Income,True,PASS
2,Bank Marketing,True,PASS
3,Breast Cancer,True,PASS
4,Metadata Created,True,PASS



✓ PART 5.3 COMPLETED SUCCESSFULLY
Ready for Part 5.4 — Automatic Metadata Detection


In [33]:
# ==============================================================================
# PART 5.4.1 — IMPORT METADATA LIBRARY
# ==============================================================================

from sdv.metadata import SingleTableMetadata

print("=" * 100)
print("PART 5.4 — AUTOMATIC METADATA DETECTION")
print("=" * 100)

print("✓ SingleTableMetadata imported successfully")

PART 5.4 — AUTOMATIC METADATA DETECTION
✓ SingleTableMetadata imported successfully


In [34]:
# ==============================================================================
# PART 5.4.2 — DETECT METADATA
# ==============================================================================

METADATA_DICT = {}

for dataset in DATASETS:

    metadata = SingleTableMetadata()

    metadata.detect_from_dataframe(

        data=DATA_DICT[dataset]

    )

    METADATA_DICT[dataset] = metadata

print("=" * 100)
print("METADATA DETECTED")
print("=" * 100)

for dataset in DATASETS:

    print(f"✓ {dataset}")

METADATA DETECTED
✓ adult_income
✓ bank_marketing
✓ breast_cancer


In [35]:
# ==============================================================================
# PART 5.4.3 — METADATA SUMMARY
# ==============================================================================

SUMMARY = []

for dataset in DATASETS:

    metadata = METADATA_DICT[dataset]

    columns = metadata.to_dict()["columns"]

    numerical = 0
    categorical = 0
    boolean = 0
    datetime = 0

    for col in columns.values():

        sdtype = col["sdtype"]

        if sdtype == "numerical":
            numerical += 1

        elif sdtype == "categorical":
            categorical += 1

        elif sdtype == "boolean":
            boolean += 1

        elif sdtype == "datetime":
            datetime += 1

    SUMMARY.append({

        "Dataset": dataset,

        "Columns": len(columns),

        "Numerical": numerical,

        "Categorical": categorical,

        "Boolean": boolean,

        "Datetime": datetime

    })

METADATA_SUMMARY_DF = pd.DataFrame(SUMMARY)

print("=" * 100)
print("METADATA SUMMARY")
print("=" * 100)

display(METADATA_SUMMARY_DF)

METADATA SUMMARY


,Dataset,Columns,Numerical,Categorical,Boolean,Datetime
0,adult_income,15,7,8,0,0
1,bank_marketing,17,7,10,0,0
2,breast_cancer,32,30,1,0,0


In [36]:
# ==============================================================================
# PART 5.4.4 — SAVE METADATA
# ==============================================================================

METADATA_DIR = os.path.join(

    TVAE_RESULT_DIR,

    "metadata"

)

os.makedirs(

    METADATA_DIR,

    exist_ok=True

)

for dataset in DATASETS:

    metadata_file = os.path.join(

        METADATA_DIR,

        f"{dataset}_metadata.json"

    )

    METADATA_DICT[dataset].save_to_json(

        metadata_file

    )

print("=" * 100)
print("METADATA FILES SAVED")
print("=" * 100)

for dataset in DATASETS:

    print(

        os.path.join(

            METADATA_DIR,

            f"{dataset}_metadata.json"

        )

    )

METADATA FILES SAVED
/content/drive/MyDrive/SPP_GAN_Project/results/TVAE/metadata/adult_income_metadata.json
/content/drive/MyDrive/SPP_GAN_Project/results/TVAE/metadata/bank_marketing_metadata.json
/content/drive/MyDrive/SPP_GAN_Project/results/TVAE/metadata/breast_cancer_metadata.json


In [37]:
# ==============================================================================
# PART 5.4.5 — VERIFICATION
# ==============================================================================

VERIFY = {

    "Metadata Objects":

        len(METADATA_DICT) == len(DATASETS),

    "Metadata Summary":

        len(METADATA_SUMMARY_DF) == len(DATASETS),

    "Metadata Directory":

        os.path.isdir(METADATA_DIR),

    "Metadata Files":

        all(

            os.path.isfile(

                os.path.join(

                    METADATA_DIR,

                    f"{dataset}_metadata.json"

                )

            )

            for dataset in DATASETS

        )

}

VERIFY_DF = pd.DataFrame({

    "Component":

        VERIFY.keys(),

    "Status":

        VERIFY.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x:

        "PASS"

        if x

        else "FAIL"

)

print("=" * 100)
print("PART 5.4 VERIFICATION")
print("=" * 100)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n" + "=" * 100)

    print("✓ PART 5.4 COMPLETED SUCCESSFULLY")

    print("Ready for Part 5.5 — TVAE Model Initialization")

    print("=" * 100)

else:

    display(

        VERIFY_DF[

            VERIFY_DF["Status"] == False

        ]

    )

    raise RuntimeError(

        "Metadata verification failed."

    )

PART 5.4 VERIFICATION


,Component,Status,Result
0,Metadata Objects,True,PASS
1,Metadata Summary,True,PASS
2,Metadata Directory,True,PASS
3,Metadata Files,True,PASS



✓ PART 5.4 COMPLETED SUCCESSFULLY
Ready for Part 5.5 — TVAE Model Initialization


In [41]:
# ==============================================================================
# PART 5.5.1 — TVAE HYPERPARAMETER CONFIGURATION
# ==============================================================================

print("=" * 100)
print("PART 5.5 — TVAE HYPERPARAMETER CONFIGURATION")
print("=" * 100)

USE_GPU = torch.cuda.is_available()

TVAE_CONFIG = {

    # --------------------------------------------------------------------------
    # Training
    # --------------------------------------------------------------------------

    "epochs": 200,

    "batch_size": 256,

    "cuda": USE_GPU,

    "verbose": True,

    # --------------------------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------------------------

    "learning_rate": 2e-4,

    "weight_decay": 1e-5,

    # --------------------------------------------------------------------------
    # Architecture
    # --------------------------------------------------------------------------

    "embedding_dim": 128,

    "compress_dims": (256, 128),

    "decompress_dims": (128, 256),

    "loss_factor": 2,

    # --------------------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------------------

    "random_state": RANDOM_STATE

}

print("✓ TVAE configuration created successfully.")

PART 5.5 — TVAE HYPERPARAMETER CONFIGURATION
✓ TVAE configuration created successfully.


In [42]:
# ==============================================================================
# PART 5.5.2 — DISPLAY TVAE CONFIGURATION
# ==============================================================================

TVAE_CONFIG_DF = pd.DataFrame({

    "Hyperparameter": TVAE_CONFIG.keys(),

    "Value": TVAE_CONFIG.values()

})

print("=" * 100)
print("TVAE HYPERPARAMETERS")
print("=" * 100)

display(TVAE_CONFIG_DF)

TVAE HYPERPARAMETERS


,Hyperparameter,Value
0,epochs,200
1,batch_size,256
2,cuda,False
3,verbose,True
4,learning_rate,0.0002
5,weight_decay,0.0000
6,embedding_dim,128
7,compress_dims,"(256, 128)"
8,decompress_dims,"(128, 256)"
9,loss_factor,2


In [43]:
# ==============================================================================
# PART 5.5.3 — SAVE TVAE CONFIGURATION
# ==============================================================================

TVAE_CONFIG_FILE = os.path.join(

    TVAE_RESULT_DIR,

    "tvae_configuration.json"

)

with open(

    TVAE_CONFIG_FILE,

    "w"

) as f:

    json.dump(

        TVAE_CONFIG,

        f,

        indent=4

    )

print("=" * 100)
print("CONFIGURATION SAVED")
print("=" * 100)

print(TVAE_CONFIG_FILE)

CONFIGURATION SAVED
/content/drive/MyDrive/SPP_GAN_Project/results/TVAE/tvae_configuration.json


In [44]:
# ==============================================================================
# PART 5.5.4 — VERIFICATION
# ==============================================================================

VERIFY = {

    "Configuration Dictionary":

        isinstance(TVAE_CONFIG, dict),

    "Configuration File":

        os.path.isfile(

            TVAE_CONFIG_FILE

        ),

    "Epochs":

        TVAE_CONFIG["epochs"] > 0,

    "Batch Size":

        TVAE_CONFIG["batch_size"] > 0,

    "Embedding Dimension":

        TVAE_CONFIG["embedding_dim"] > 0,

    "Compress Layers":

        len(TVAE_CONFIG["compress_dims"]) > 0,

    "Decompress Layers":

        len(TVAE_CONFIG["decompress_dims"]) > 0

}

VERIFY_DF = pd.DataFrame({

    "Component": VERIFY.keys(),

    "Status": VERIFY.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x: "PASS" if x else "FAIL"

)

print("=" * 100)
print("PART 5.5 VERIFICATION")
print("=" * 100)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n" + "=" * 100)

    print("✓ PART 5.5 COMPLETED SUCCESSFULLY")

    print("Ready for Part 5.6 — TVAE Model Initialization")

    print("=" * 100)

else:

    display(

        VERIFY_DF[

            VERIFY_DF["Status"] == False

        ]

    )

    raise RuntimeError(

        "TVAE configuration verification failed."

    )

PART 5.5 VERIFICATION


,Component,Status,Result
0,Configuration Dictionary,True,PASS
1,Configuration File,True,PASS
2,Epochs,True,PASS
3,Batch Size,True,PASS
4,Embedding Dimension,True,PASS
5,Compress Layers,True,PASS
6,Decompress Layers,True,PASS



✓ PART 5.5 COMPLETED SUCCESSFULLY
Ready for Part 5.6 — TVAE Model Initialization


In [45]:
# ==============================================================================
# PART 5.6.1 — IMPORT TVAE MODEL
# ==============================================================================

print("=" * 100)
print("PART 5.6.1 — IMPORT TVAE MODEL")
print("=" * 100)

# ==============================================================================
# Import TVAE
# ==============================================================================

try:

    from sdv.single_table import TVAESynthesizer

    TVAE_IMPORTED = True

    print("✓ TVAESynthesizer imported successfully")

except Exception as e:

    TVAE_IMPORTED = False

    print("✗ Failed to import TVAESynthesizer")
    print(e)

# ==============================================================================
# Display Version Information
# ==============================================================================

print("\nInstalled Library Versions")

print("-" * 100)

try:

    import sdv

    print(f"SDV Version          : {sdv.__version__}")

except:

    print("SDV Version          : Not Available")

try:

    import torch

    print(f"PyTorch Version      : {torch.__version__}")

except:

    print("PyTorch Version      : Not Available")

# ==============================================================================
# GPU Information
# ==============================================================================

print("\nDevice Information")

print("-" * 100)

if torch.cuda.is_available():

    DEVICE = "cuda"

    print(f"GPU Available        : Yes")

    print(f"GPU Name             : {torch.cuda.get_device_name(0)}")

else:

    DEVICE = "cpu"

    print("GPU Available        : No")
    print("Using CPU")

# ==============================================================================
# Verification Table
# ==============================================================================

TVAE_IMPORT_VERIFY = pd.DataFrame({

    "Component":[

        "TVAESynthesizer Imported",
        "SDV Installed",
        "PyTorch Installed"

    ],

    "Status":[

        TVAE_IMPORTED,

        "sdv" in globals(),

        "torch" in globals()

    ]

})

TVAE_IMPORT_VERIFY["Result"] = TVAE_IMPORT_VERIFY["Status"].map(

    lambda x: "PASS" if x else "FAIL"

)

display(TVAE_IMPORT_VERIFY)

# ==============================================================================
# Final Verification
# ==============================================================================

if TVAE_IMPORT_VERIFY["Status"].all():

    print("\n✓ PART 5.6.1 Completed Successfully")

else:

    raise RuntimeError(

        "TVAE import failed. Please check SDV installation."

    )

PART 5.6.1 — IMPORT TVAE MODEL
✓ TVAESynthesizer imported successfully

Installed Library Versions
----------------------------------------------------------------------------------------------------
SDV Version          : 1.37.3
PyTorch Version      : 2.11.0+cpu

Device Information
----------------------------------------------------------------------------------------------------
GPU Available        : No
Using CPU


,Component,Status,Result
0,TVAESynthesizer Imported,True,PASS
1,SDV Installed,True,PASS
2,PyTorch Installed,True,PASS



✓ PART 5.6.1 Completed Successfully


In [47]:
# ==============================================================================
# PART 5.6.2 — INITIALIZE TVAE MODELS
# ==============================================================================

print("=" * 100)
print("PART 5.6.2 — INITIALIZE TVAE MODELS")
print("=" * 100)

TVAE_MODELS = {}

for dataset in DATASETS:

    print(f"\nInitializing TVAE Model : {dataset}")

    TVAE_MODELS[dataset] = TVAESynthesizer(

        metadata=METADATA_DICT[dataset],

        epochs=TVAE_CONFIG["epochs"],

        batch_size=TVAE_CONFIG["batch_size"],

        embedding_dim=TVAE_CONFIG["embedding_dim"],

        compress_dims=TVAE_CONFIG["compress_dims"],

        decompress_dims=TVAE_CONFIG["decompress_dims"],

        # Removed l2scale as it's not in TVAE_CONFIG
        loss_factor=TVAE_CONFIG["loss_factor"],

        cuda=TVAE_CONFIG["cuda"],

        verbose=True

    )

print("\n✓ All TVAE models initialized successfully.")

PART 5.6.2 — INITIALIZE TVAE MODELS

Initializing TVAE Model : adult_income

Initializing TVAE Model : bank_marketing

Initializing TVAE Model : breast_cancer

✓ All TVAE models initialized successfully.


In [48]:
# ==============================================================================
# VERIFY TVAE INITIALIZATION
# ==============================================================================

VERIFY_TVAE = pd.DataFrame({

    "Dataset": DATASETS,

    "Initialized": [

        dataset in TVAE_MODELS

        for dataset in DATASETS

    ]

})

VERIFY_TVAE["Result"] = VERIFY_TVAE["Initialized"].map(

    lambda x: "PASS" if x else "FAIL"

)

display(VERIFY_TVAE)

if VERIFY_TVAE["Initialized"].all():

    print("\n✓ PART 5.6.2 Completed Successfully")

else:

    raise RuntimeError("TVAE initialization failed.")

,Dataset,Initialized,Result
0,adult_income,True,PASS
1,bank_marketing,True,PASS
2,breast_cancer,True,PASS



✓ PART 5.6.2 Completed Successfully


In [49]:
# ==============================================================================
# PART 5.6.3 — TVAE MODEL CONFIGURATION SUMMARY
# ==============================================================================

print("=" * 100)
print("PART 5.6.3 — TVAE MODEL CONFIGURATION SUMMARY")
print("=" * 100)

TVAE_CONFIGURATION_SUMMARY = []

for dataset in DATASETS:

    df = DATA_DICT[dataset]

    TVAE_CONFIGURATION_SUMMARY.append({

        "Dataset":
            dataset,

        "Rows":
            df.shape[0],

        "Columns":
            df.shape[1],

        "Embedding Dimension":
            TVAE_CONFIG["embedding_dim"],

        "Compress Dimensions":
            str(TVAE_CONFIG["compress_dims"]),

        "Decompress Dimensions":
            str(TVAE_CONFIG["decompress_dims"]),

        "Batch Size":
            TVAE_CONFIG["batch_size"],

        "Epochs":
            TVAE_CONFIG["epochs"],

        "Learning Rate":
            TVAE_CONFIG["learning_rate"],

        "Weight Decay":
            TVAE_CONFIG["weight_decay"],

        "Loss Factor":
            TVAE_CONFIG["loss_factor"],

        "Device":
            "GPU" if TVAE_CONFIG["cuda"] else "CPU"

    })

TVAE_CONFIGURATION_DF = pd.DataFrame(

    TVAE_CONFIGURATION_SUMMARY

)

display(TVAE_CONFIGURATION_DF)

PART 5.6.3 — TVAE MODEL CONFIGURATION SUMMARY


,Dataset,Rows,Columns,Embedding Dimension,Compress Dimensions,Decompress Dimensions,Batch Size,Epochs,Learning Rate,Weight Decay,Loss Factor,Device
0,adult_income,32510,15,128,"(256, 128)","(128, 256)",256,200,0.0002,0.0000,2,CPU
1,bank_marketing,45211,17,128,"(256, 128)","(128, 256)",256,200,0.0002,0.0000,2,CPU
2,breast_cancer,569,32,128,"(256, 128)","(128, 256)",256,200,0.0002,0.0000,2,CPU


In [50]:
# ==============================================================================
# SAVE TVAE CONFIGURATION SUMMARY
# ==============================================================================

CONFIG_SUMMARY_FILE = os.path.join(

    TVAE_REPORT_DIR,

    "tvae_configuration_summary.csv"

)

TVAE_CONFIGURATION_DF.to_csv(

    CONFIG_SUMMARY_FILE,

    index=False

)

print("Configuration summary saved to:")

print(CONFIG_SUMMARY_FILE)

Configuration summary saved to:
/content/drive/MyDrive/SPP_GAN_Project/reports/TVAE/tvae_configuration_summary.csv


In [51]:
# ==============================================================================
# PART 5.6.3 VERIFICATION
# ==============================================================================

VERIFY = {

    "Configuration Table":

        len(TVAE_CONFIGURATION_DF) == len(DATASETS),

    "Summary Saved":

        os.path.isfile(CONFIG_SUMMARY_FILE)

}

VERIFY_DF = pd.DataFrame({

    "Component":

        VERIFY.keys(),

    "Status":

        VERIFY.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x: "PASS" if x else "FAIL"

)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n✓ PART 5.6.3 Completed Successfully")

else:

    raise RuntimeError(

        "Configuration summary verification failed."

    )

,Component,Status,Result
0,Configuration Table,True,PASS
1,Summary Saved,True,PASS



✓ PART 5.6.3 Completed Successfully


In [ ]:
# ==============================================================================
# PART 5.6.4 — TVAE TRAINING
# ==============================================================================

import time

print("=" * 100)
print("PART 5.6.4 — TVAE TRAINING")
print("=" * 100)

TVAE_TRAINING_RESULTS = []

TVAE_START_TIME = time.time()

for dataset in DATASETS:

    print("\n" + "=" * 100)
    print(f"Training TVAE : {dataset}")
    print("=" * 100)

    train_start = time.time()

    try:

        # ----------------------------------------------------------------------
        # Load Dataset
        # ----------------------------------------------------------------------

        train_df = DATA_DICT[dataset].copy()

        print(f"Dataset Shape : {train_df.shape}")

        # ----------------------------------------------------------------------
        # Train TVAE
        # ----------------------------------------------------------------------

        TVAE_MODELS[dataset].fit(

            train_df

        )

        train_end = time.time()

        elapsed = train_end - train_start

        status = "Success"

        print(f"\n✓ Training Completed")

        print(f"Training Time : {elapsed:.2f} seconds")

    except Exception as e:

        train_end = time.time()

        elapsed = train_end - train_start

        status = "Failed"

        print("\n✗ Training Failed")

        print(e)

    # --------------------------------------------------------------------------
    # Store Results
    # --------------------------------------------------------------------------

    TVAE_TRAINING_RESULTS.append({

        "Dataset":

            dataset,

        "Rows":

            train_df.shape[0] if status == "Success" else np.nan,

        "Columns":

            train_df.shape[1] if status == "Success" else np.nan,

        "Epochs":

            TVAE_CONFIG["epochs"],

        "Batch Size":

            TVAE_CONFIG["batch_size"],

        "Embedding Dimension":

            TVAE_CONFIG["embedding_dim"],

        "Training Time (Seconds)":

            round(elapsed,2),

        "Status":

            status

    })

TVAE_TOTAL_TIME = time.time() - TVAE_START_TIME

print("\n" + "=" * 100)

print(f"Total TVAE Training Time : {TVAE_TOTAL_TIME:.2f} seconds")

print("=" * 100)

PART 5.6.4 — TVAE TRAINING

Training TVAE : adult_income
Dataset Shape : (32510, 15)


Loss: -18.40:  68%|██████▊   | 135/200 [15:38<15:53, 14.67s/it]

In [ ]:
# ==============================================================================
# CREATE TRAINING SUMMARY
# ==============================================================================

TVAE_TRAINING_SUMMARY_DF = pd.DataFrame(

    TVAE_TRAINING_RESULTS

)

display(TVAE_TRAINING_SUMMARY_DF)

STATISTICAL PRIOR MODULE
Latent Shape : torch.Size([8, 128])
Prior Shape  : torch.Size([8, 64])
Output Shape : torch.Size([8, 192])


In [ ]:
# ==============================================================================
# SAVE TRAINING SUMMARY
# ==============================================================================

TRAINING_SUMMARY_FILE = os.path.join(

    TVAE_REPORT_DIR,

    "tvae_training_summary.csv"

)

TVAE_TRAINING_SUMMARY_DF.to_csv(

    TRAINING_SUMMARY_FILE,

    index=False

)

print("Training summary saved to:")

print(TRAINING_SUMMARY_FILE)

Temporary tensors removed.


In [ ]:
# ==============================================================================
# PART 5.6.4 VERIFICATION
# ==============================================================================

VERIFY = {

    "Three Datasets Trained":

        len(TVAE_TRAINING_SUMMARY_DF) == len(DATASETS),

    "Training Summary Saved":

        os.path.isfile(TRAINING_SUMMARY_FILE),

    "All Training Successful":

        (TVAE_TRAINING_SUMMARY_DF["Status"] == "Success").all()

}

VERIFY_DF = pd.DataFrame({

    "Component":

        VERIFY.keys(),

    "Status":

        VERIFY.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x: "PASS" if x else "FAIL"

)

print("=" * 100)
print("PART 5.6.4 VERIFICATION")
print("=" * 100)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n✓ PART 5.6.4 Completed Successfully")

else:

    print("\n⚠ Some datasets failed during training. Check the error messages above.")

SECTION D COMPLETED

✓ Gaussian Priors Loaded
✓ Copula Priors Loaded
✓ Prior Encoder
✓ Prior Fusion Layer
✓ Statistical Prior Module

NEXT
--------------------------------------------------
Section E : Generator Architecture



In [ ]:
# ==============================================================================
# PART 5.6.5 — SAVE TRAINED TVAE MODELS
# ==============================================================================

import os
import joblib

print("=" * 100)
print("PART 5.6.5 — SAVE TRAINED TVAE MODELS")
print("=" * 100)

TVAE_MODEL_FILES = {}

os.makedirs(TVAE_MODEL_DIR, exist_ok=True)

for dataset in DATASETS:

    print(f"\nSaving TVAE Model : {dataset}")

    model_file = os.path.join(

        TVAE_MODEL_DIR,

        f"{dataset}_tvae.pkl"

    )

    try:

        joblib.dump(

            TVAE_MODELS[dataset],

            model_file

        )

        TVAE_MODEL_FILES[dataset] = model_file

        print("✓ Model Saved")

        print(model_file)

    except Exception as e:

        TVAE_MODEL_FILES[dataset] = None

        print("✗ Saving Failed")

        print(e)

ResidualBlock created.


In [ ]:
# ==============================================================================
# CREATE SAVED MODEL SUMMARY
# ==============================================================================

TVAE_MODEL_SUMMARY = []

for dataset in DATASETS:

    model_file = TVAE_MODEL_FILES.get(dataset)

    exists = False
    size_mb = np.nan

    if model_file is not None:

        exists = os.path.isfile(model_file)

        if exists:

            size_mb = round(

                os.path.getsize(model_file) /

                (1024 * 1024),

                2

            )

    TVAE_MODEL_SUMMARY.append({

        "Dataset":

            dataset,

        "Model File":

            model_file,

        "Exists":

            exists,

        "Model Size (MB)":

            size_mb

    })

TVAE_MODEL_SUMMARY_DF = pd.DataFrame(

    TVAE_MODEL_SUMMARY

)

display(TVAE_MODEL_SUMMARY_DF)

NoiseFusion created.


In [ ]:
# ==============================================================================
# SAVE MODEL INVENTORY
# ==============================================================================

MODEL_SUMMARY_FILE = os.path.join(

    TVAE_REPORT_DIR,

    "tvae_saved_models_summary.csv"

)

TVAE_MODEL_SUMMARY_DF.to_csv(

    MODEL_SUMMARY_FILE,

    index=False

)

print("Model inventory saved to:")

print(MODEL_SUMMARY_FILE)

Generator created.


In [ ]:
# ==============================================================================
# PART 5.6.5 VERIFICATION
# ==============================================================================

VERIFY = {

    "Three Models Saved":

        len(TVAE_MODEL_SUMMARY_DF) == len(DATASETS),

    "All Model Files Exist":

        TVAE_MODEL_SUMMARY_DF["Exists"].all(),

    "Inventory Saved":

        os.path.isfile(MODEL_SUMMARY_FILE)

}

VERIFY_DF = pd.DataFrame({

    "Component":

        VERIFY.keys(),

    "Status":

        VERIFY.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x: "PASS" if x else "FAIL"

)

print("=" * 100)
print("PART 5.6.5 VERIFICATION")
print("=" * 100)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n✓ PART 5.6.5 Completed Successfully")

else:

    raise RuntimeError(

        "TVAE model saving verification failed."

    )

Generator factory created.


In [ ]:
# ==============================================================================
# PART 5.7.1 — GENERATE SYNTHETIC DATA
# ==============================================================================

print("=" * 100)
print("PART 5.7.1 — GENERATE SYNTHETIC DATA")
print("=" * 100)

TVAE_SYNTHETIC_DATA = {}

TVAE_GENERATION_RESULTS = []

for dataset in DATASETS:

    print("\n" + "=" * 100)
    print(f"Generating Synthetic Data : {dataset}")
    print("=" * 100)

    real_df = DATA_DICT[dataset]

    n_rows = len(real_df)

    print(f"Rows to Generate : {n_rows}")

    synthetic_df = TVAE_MODELS[dataset].sample(

        num_rows=n_rows

    )

    TVAE_SYNTHETIC_DATA[dataset] = synthetic_df

    TVAE_GENERATION_RESULTS.append({

        "Dataset":

            dataset,

        "Real Rows":

            len(real_df),

        "Synthetic Rows":

            len(synthetic_df),

        "Columns":

            synthetic_df.shape[1]

    })

    print("✓ Synthetic Dataset Generated")

    print(f"Shape : {synthetic_df.shape}")

GENERATOR VERIFICATION
Noise Shape      : torch.Size([8, 128])
Prior Shape      : torch.Size([8, 64])
Generated Shape  : torch.Size([8, 111])


In [ ]:
# ==============================================================================
# SAVE SYNTHETIC DATASETS
# ==============================================================================

print("=" * 100)
print("SAVING SYNTHETIC DATASETS")
print("=" * 100)

TVAE_SYNTHETIC_FILES = {}

os.makedirs(TVAE_RESULT_DIR, exist_ok=True)

for dataset in DATASETS:

    save_file = os.path.join(

        TVAE_RESULT_DIR,

        f"{dataset}_tvae_synthetic.csv"

    )

    TVAE_SYNTHETIC_DATA[dataset].to_csv(

        save_file,

        index=False

    )

    TVAE_SYNTHETIC_FILES[dataset] = save_file

    print(f"✓ Saved : {dataset}")

Temporary tensors removed.


In [ ]:
# ==============================================================================
# SYNTHETIC DATA SUMMARY
# ==============================================================================

TVAE_SYNTHETIC_SUMMARY_DF = pd.DataFrame(

    TVAE_GENERATION_RESULTS

)

display(TVAE_SYNTHETIC_SUMMARY_DF)

SUMMARY_FILE = os.path.join(

    TVAE_REPORT_DIR,

    "tvae_synthetic_dataset_summary.csv"

)

TVAE_SYNTHETIC_SUMMARY_DF.to_csv(

    SUMMARY_FILE,

    index=False

)

print("\nSummary Saved To:")

print(SUMMARY_FILE)

SECTION E COMPLETED

✓ Residual Blocks
✓ Prior Conditioning
✓ Noise Fusion
✓ Generator Network
✓ Output Layer
✓ Generator Factory
✓ Generator Verification

NEXT
--------------------------------------------------
Section F : Discriminator Architecture



In [ ]:
# ==============================================================================
# PREVIEW SYNTHETIC DATASETS
# ==============================================================================

for dataset in DATASETS:

    print("\n" + "=" * 100)
    print(dataset.upper())
    print("=" * 100)

    display(

        TVAE_SYNTHETIC_DATA[dataset].head()

    )

Discriminator residual block created.


In [ ]:
# ==============================================================================
# PART 5.7 VERIFICATION
# ==============================================================================

VERIFY = {

    "Three Synthetic Datasets":

        len(TVAE_SYNTHETIC_DATA) == len(DATASETS),

    "Files Saved":

        all(

            os.path.isfile(file)

            for file in TVAE_SYNTHETIC_FILES.values()

        ),

    "Equal Number of Rows":

        all(

            len(DATA_DICT[d]) == len(TVAE_SYNTHETIC_DATA[d])

            for d in DATASETS

        ),

    "Equal Number of Columns":

        all(

            DATA_DICT[d].shape[1] == TVAE_SYNTHETIC_DATA[d].shape[1]

            for d in DATASETS

        )

}

VERIFY_DF = pd.DataFrame({

    "Component":

        VERIFY.keys(),

    "Status":

        VERIFY.values()

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x: "PASS" if x else "FAIL"

)

print("=" * 100)
print("PART 5.7 VERIFICATION")
print("=" * 100)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n✓ PART 5.7 Completed Successfully")

else:

    raise RuntimeError(

        "Synthetic data generation verification failed."

    )

Feature extractor created.


In [ ]:
# ==============================================================================
# PART 5.8.1 — CREATE OUTPUT DIRECTORIES
# ==============================================================================

print("="*100)
print("PART 5.8.1 — CREATE OUTPUT DIRECTORIES")
print("="*100)

import os

TVAE_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "TVAE"
)

TVAE_SYNTHETIC_DIR = os.path.join(
    PROJECT_DIR,
    "synthetic_data",
    "TVAE"
)

os.makedirs(TVAE_MODEL_DIR, exist_ok=True)
os.makedirs(TVAE_SYNTHETIC_DIR, exist_ok=True)

print("Model Directory")
print(TVAE_MODEL_DIR)

print("\nSynthetic Data Directory")
print(TVAE_SYNTHETIC_DIR)

Discriminator created.


In [ ]:
# ==============================================================================
# PART 5.8.2 — SAVE TRAINED TVAE MODELS
# ==============================================================================

print("="*100)
print("PART 5.8.2 — SAVE TRAINED TVAE MODELS")
print("="*100)

MODEL_SAVE_LOG = []

for dataset in DATASETS:

    model = TVAE_MODELS[dataset]

    model_path = os.path.join(
        TVAE_MODEL_DIR,
        f"{dataset}_TVAE.pkl"
    )

    model.save(model_path)

    MODEL_SAVE_LOG.append({

        "Dataset": dataset,
        "Model_File": os.path.basename(model_path),
        "Saved": os.path.exists(model_path)

    })

    print(f"{dataset:<20} ✓ Saved")

MODEL_SAVE_DF = pd.DataFrame(MODEL_SAVE_LOG)

display(MODEL_SAVE_DF)

Discriminator factory created.


In [ ]:
# ==============================================================================
# PART 5.8.3 — SAVE SYNTHETIC DATASETS
# ==============================================================================

print("="*100)
print("PART 5.8.3 — SAVE SYNTHETIC DATASETS")
print("="*100)

DATA_SAVE_LOG = []

for dataset in DATASETS:

    synthetic_df = TVAE_SYNTHETIC_DATA[dataset]

    csv_path = os.path.join(
        TVAE_SYNTHETIC_DIR,
        f"{dataset}_TVAE_synthetic.csv"
    )

    synthetic_df.to_csv(
        csv_path,
        index=False
    )

    DATA_SAVE_LOG.append({

        "Dataset": dataset,
        "Rows": len(synthetic_df),
        "Columns": synthetic_df.shape[1],
        "Saved": os.path.exists(csv_path)

    })

    print(f"{dataset:<20} ✓ Saved")

SYNTHETIC_SAVE_DF = pd.DataFrame(DATA_SAVE_LOG)

display(SYNTHETIC_SAVE_DF)

DISCRIMINATOR VERIFICATION
Input Shape        : torch.Size([8, 111])
Prediction Shape   : torch.Size([8, 1])
Feature Shape      : torch.Size([8, 128])


In [ ]:
# ==============================================================================
# PART 5.8.4 — FINAL VERIFICATION
# ==============================================================================

print("="*100)
print("PART 5.8.4 — FINAL VERIFICATION")
print("="*100)

VERIFY_RESULTS = []

for dataset in DATASETS:

    model_file = os.path.join(
        TVAE_MODEL_DIR,
        f"{dataset}_TVAE.pkl"
    )

    synthetic_file = os.path.join(
        TVAE_SYNTHETIC_DIR,
        f"{dataset}_TVAE_synthetic.csv"
    )

    VERIFY_RESULTS.append({

        "Dataset": dataset,

        "Model Saved":
            os.path.exists(model_file),

        "Synthetic Saved":
            os.path.exists(synthetic_file)

    })

VERIFY_DF = pd.DataFrame(VERIFY_RESULTS)

VERIFY_DF["Status"] = (
    VERIFY_DF["Model Saved"] &
    VERIFY_DF["Synthetic Saved"]
)

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(
    lambda x: "PASS" if x else "FAIL"
)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n✓ PART 5.8 Completed Successfully")

else:

    raise RuntimeError(
        "Some TVAE models or synthetic datasets were not saved correctly."
    )

Temporary tensors removed.


In [ ]:
# ==============================================================================
# PART 5.9.1 — VERIFY SAVED TVAE MODELS
# ==============================================================================

print("=" * 100)
print("VERIFY TRAINED TVAE MODELS")
print("=" * 100)

MODEL_VERIFY = []

for dataset in DATASETS:

    model_path = os.path.join(
        TVAE_MODEL_DIR,
        f"{dataset}_tvae.pkl"
    )

    MODEL_VERIFY.append({

        "Dataset": dataset,
        "Model File": os.path.basename(model_path),
        "Exists": os.path.exists(model_path)

    })

MODEL_VERIFY_DF = pd.DataFrame(MODEL_VERIFY)

display(MODEL_VERIFY_DF)

SECTION F COMPLETED

✓ Feature Extractor
✓ Residual Layers
✓ Real/Fake Classifier
✓ Feature Embedding Output
✓ Discriminator Factory
✓ Verification Successful

NEXT
--------------------------------------------------
Section G : Loss Functions

Adversarial Loss Ready


In [ ]:
# ==============================================================================
# PART 5.9.2 — VERIFY SYNTHETIC DATASETS
# ==============================================================================

print("=" * 100)
print("VERIFY SYNTHETIC DATASETS")
print("=" * 100)

DATA_VERIFY = []

for dataset in DATASETS:

    data_path = os.path.join(
        TVAE_SYNTHETIC_DIR,
        f"{dataset}_synthetic.csv"
    )

    if os.path.exists(data_path):

        df = pd.read_csv(data_path)

        rows = len(df)
        cols = len(df.columns)

    else:

        rows = 0
        cols = 0

    DATA_VERIFY.append({

        "Dataset": dataset,
        "Rows": rows,
        "Columns": cols,
        "Exists": os.path.exists(data_path)

    })

DATA_VERIFY_DF = pd.DataFrame(DATA_VERIFY)

display(DATA_VERIFY_DF)

Adversarial loss created.


In [ ]:
# ==============================================================================
# PART 5.9.3 — TVAE TRAINING SUMMARY
# ==============================================================================

print("=" * 100)
print("TVAE TRAINING SUMMARY")
print("=" * 100)

SUMMARY = []

for dataset in DATASETS:

    summary = {

        "Dataset": dataset,

        "Model Saved":
            os.path.exists(
                os.path.join(
                    TVAE_MODEL_DIR,
                    f"{dataset}_tvae.pkl"
                )
            ),

        "Synthetic Data Saved":
            os.path.exists(
                os.path.join(
                    TVAE_SYNTHETIC_DIR,
                    f"{dataset}_synthetic.csv"
                )
            ),

        "Metadata Available":
            dataset in METADATA_DICT

    }

    SUMMARY.append(summary)

SUMMARY_DF = pd.DataFrame(SUMMARY)

display(SUMMARY_DF)

Prior consistency loss created.


In [ ]:
# ==============================================================================
# PART 5.9.4 — FINAL VERIFICATION
# ==============================================================================

print("=" * 100)
print("NOTEBOOK 05 FINAL VERIFICATION")
print("=" * 100)

VERIFY_DF = pd.DataFrame({

    "Component": [

        "Configuration Loaded",
        "Datasets Loaded",
        "Metadata Created",
        "TVAE Models Trained",
        "Models Saved",
        "Synthetic Data Generated",
        "Synthetic Data Saved"

    ],

    "Status": [

        True,

        len(DATA_DICT) == len(DATASETS),

        len(METADATA_DICT) == len(DATASETS),

        MODEL_VERIFY_DF["Exists"].all(),

        MODEL_VERIFY_DF["Exists"].all(),

        DATA_VERIFY_DF["Exists"].all(),

        DATA_VERIFY_DF["Exists"].all()

    ]

})

VERIFY_DF["Result"] = VERIFY_DF["Status"].map(

    lambda x: "PASS" if x else "FAIL"

)

display(VERIFY_DF)

if VERIFY_DF["Status"].all():

    print("\n✓ NOTEBOOK 05 COMPLETED SUCCESSFULLY")

    print("\nTVAE models trained for all datasets.")

    print("Synthetic datasets generated successfully.")

    print("Notebook is ready for Notebook 06 (DP-CTGAN).")

else:

    raise RuntimeError(
        "Notebook 05 verification failed."
    )

Feature matching loss created.
